# 06 — KernelSHAP for Full Hybrid Model

Model-agnostic SHAP using `shap.KernelExplainer` on the **entire** hybrid ensemble (Ridge + XGBoost + LightGBM + RandomForest + meta-learner), not just the XGBoost component.

**Prereqs:** Run **02_hybrid_ensemble.ipynb** so `03_ml_layer_hybrid/artifacts/hybrid_cluster_bundle.joblib` exists.

**Data:** Full feature table from **`hf_data/.../outputs/`** or **`data/feature_data/02_feature_layer/training/outputs/`**.

**Trade-offs:**
- **Pros:** Explains the full hybrid prediction including meta-learner weighting
- **Cons:** Slow (~1-5 min per row) due to sampling-based approach

**Outputs:** `03_ml_layer_hybrid/artifacts/hybrid_xai/kernel_shap_*.joblib`

In [1]:
%pip install -q numpy pandas scikit-learn xgboost lightgbm shap matplotlib joblib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlxtend 0.24.0 requires joblib>=1.5.2, but you have joblib 1.4.2 which is incompatible.
mlxtend 0.24.0 requires numpy>=2.3.5, but you have numpy 2.2.6 which is incompatible.
mlxtend 0.24.0 requires scipy>=1.16.3, but you have scipy 1.13.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_percentage_error

warnings.filterwarnings("ignore")

import sys

_HERE = Path.cwd().resolve()


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "feature_data").is_dir() or (p / "hf_data").is_dir():
            return p
    return start.parent


REPO_ROOT = _repo_root(_HERE)

_FEATURE_OUTPUT_CANDIDATES = [
    REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs",
    REPO_ROOT / "data" / "feature_data" / "02_feature_layer" / "training" / "outputs",
]


def _feature_outputs_dir() -> Path:
    for d in _FEATURE_OUTPUT_CANDIDATES:
        if d.is_dir() and any(d.glob("hdb_feature_table_*.csv")):
            return d
    tried = "\n  ".join(str(d) for d in _FEATURE_OUTPUT_CANDIDATES)
    raise FileNotFoundError(
        "No hdb_feature_table_*.csv found. Run notebooks/00_download_data_from_HF.ipynb "
        "or place CSVs under one of:\n  " + tried
    )


def _hybrid_ml_dir(repo: Path) -> Path:
    candidates = [
        repo / "notebooks" / "03_ml_layer_hybrid",
        repo / "03_ml_layer_hybrid",
    ]
    for d in candidates:
        if (d / "yc_hybrid_inference.py").is_file():
            return d
    raise FileNotFoundError(
        "yc_hybrid_inference.py not found. Expected under notebooks/03_ml_layer_hybrid/."
    )


HYBRID_DIR = _hybrid_ml_dir(REPO_ROOT)
if str(HYBRID_DIR) not in sys.path:
    sys.path.insert(0, str(HYBRID_DIR))

import yc_hybrid_inference
importlib.reload(yc_hybrid_inference)
from yc_hybrid_inference import predict_price, load_bundle

HF_DATA_ROOT = _feature_outputs_dir()


def _latest_feature_snapshot_date(root: Path) -> str:
    tables = sorted(root.glob("hdb_feature_table_*.csv"))
    if not tables:
        raise FileNotFoundError(f"No hdb_feature_table_*.csv under {root}")
    return tables[-1].stem.split("_")[-1]


_snap = _latest_feature_snapshot_date(HF_DATA_ROOT)
all_path = HF_DATA_ROOT / f"hdb_feature_table_{_snap}.csv"

HERE = REPO_ROOT
ART = HYBRID_DIR / "artifacts"
OUT_DIR = ART / "hybrid_xai"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = ART / "hybrid_cluster_bundle.joblib"

TARGET = "resale_price"
YEAR_COL = "transaction_year"

print("HF_DATA_ROOT:", HF_DATA_ROOT)
print("Feature table:", all_path.name)
print("Artifacts:", ART)
print("OUT_DIR:", OUT_DIR)

HF_DATA_ROOT: /Users/bhuvesh/Documents/PropertyLens/data/feature_data/02_feature_layer/training/outputs
Feature table: hdb_feature_table_20260412.csv
Artifacts: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts
OUT_DIR: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai


/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 — Load data and hybrid bundle

In [3]:
df = pd.read_csv(all_path)
df = df.sort_values([YEAR_COL, "address_key"], kind="mergesort").reset_index(drop=True)

bundle = joblib.load(BUNDLE_PATH)
FEATURE_COLS = bundle["feature_columns"]
N_CLUSTERS = int(bundle["n_clusters"])

train_mask = df[YEAR_COL] < 2024
val_mask = df[YEAR_COL] == 2024
test_mask = df[YEAR_COL] >= 2025

X_all = df[FEATURE_COLS].fillna(0).astype(float)
y_all = df[TARGET].astype(float)

X_train = X_all.loc[train_mask].values
y_train = y_all.loc[train_mask].values
X_test = X_all.loc[test_mask].values
y_test = y_all.loc[test_mask].values

print("Train rows:", len(X_train), "| Test rows:", len(X_test), "| Features:", len(FEATURE_COLS))

Train rows: 205930 | Test rows: 29266 | Features: 70


## 2 — Create hybrid predict function wrapper

KernelExplainer requires a callable that takes a 2D array and returns predictions. We wrap `predict_price()` from `yc_hybrid_inference`.

In [4]:
def hybrid_predict_fn(X):
    """Wrapper for predict_price that handles array conversion."""
    return predict_price(np.asarray(X, dtype=float), bundle)


# Sanity check: verify the wrapper works
_nq = min(100, len(X_test))
_test_p = hybrid_predict_fn(X_test[:_nq])
_mape = mean_absolute_percentage_error(y_test[:_nq], _test_p) * 100
print(f"Hybrid MAPE on first {_nq} test rows: {_mape:.2f}%")

Hybrid MAPE on first 100 test rows: 6.03%


## 3 — Create background dataset and KernelExplainer

KernelSHAP uses a background dataset to estimate feature expectations. We use k-means clustering to summarize the training data into representative centroids.

In [5]:
N_BACKGROUND = 50  # Number of k-means centroids for background

print(f"Creating background dataset with {N_BACKGROUND} k-means centroids...")
t0 = time.time()

# Subsample training data for k-means (for speed)
rng = np.random.RandomState(42)
train_sample_idx = rng.choice(len(X_train), min(10000, len(X_train)), replace=False)
X_train_sample = X_train[train_sample_idx]

# Create k-means summarized background
background = shap.kmeans(X_train_sample, N_BACKGROUND)

print(f"Background dataset created in {time.time() - t0:.1f}s")
print(f"Background shape: {background.data.shape}")

Creating background dataset with 50 k-means centroids...
Background dataset created in 0.9s
Background shape: (50, 70)


In [6]:
print("Creating KernelExplainer...")
t0 = time.time()

explainer = shap.KernelExplainer(hybrid_predict_fn, background)

print(f"KernelExplainer created in {time.time() - t0:.1f}s")
print(f"Expected value (base prediction): ${explainer.expected_value:,.0f}")

Creating KernelExplainer...
KernelExplainer created in 0.8s
Expected value (base prediction): $461,092


## 4 — Compute SHAP values for test samples

KernelSHAP is slow (~1-5 min per row), so we compute SHAP values for a small sample of test rows with timing benchmarks.

In [7]:
N_SAMPLES = 20  # Number of test samples to explain (keep small due to speed)
N_SHAP_SAMPLES = 500  # Number of samples for SHAP estimation per row

# Select random test samples
test_sample_idx = rng.choice(len(X_test), N_SAMPLES, replace=False)
X_sample = X_test[test_sample_idx]
y_sample = y_test[test_sample_idx]

print(f"Computing KernelSHAP values for {N_SAMPLES} test samples...")
print(f"Using nsamples={N_SHAP_SAMPLES} per row")
print("-" * 50)

t0 = time.time()
shap_values = explainer.shap_values(X_sample, nsamples=N_SHAP_SAMPLES, silent=True)
elapsed = time.time() - t0

print(f"\nCompleted in {elapsed:.1f}s ({elapsed/N_SAMPLES:.1f}s per row)")
print(f"SHAP values shape: {shap_values.shape}")

Computing KernelSHAP values for 20 test samples...
Using nsamples=500 per row
--------------------------------------------------

Completed in 7979.1s (399.0s per row)
SHAP values shape: (20, 70)


## 5 — Visualizations

### 5.1 Summary plot (global feature importance)

In [8]:
# Global feature importance (mean |SHAP|)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = dict(zip(FEATURE_COLS, mean_abs_shap))
feature_importance = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))

print("Top 10 features by mean |SHAP| (KernelSHAP on full hybrid):")
for i, (feat, imp) in enumerate(list(feature_importance.items())[:10]):
    print(f"  {i+1:2d}. {feat:35s} ${imp:>12,.0f}")

Top 10 features by mean |SHAP| (KernelSHAP on full hybrid):
   1. transaction_year                    $     190,764
   2. floor_area_sqm                      $      62,550
   3. lease_remaining_years               $      46,842
   4. room_count                          $      29,954
   5. dist_to_highway_m                   $      21,034
   6. mall_weighted_access_3km            $      19,622
   7. level_mid                           $      18,428
   8. mall_count_3km                      $      14,905
   9. dist_to_mrt_m                       $       9,991
  10. dist_to_nearest_mall_m              $       5,431


In [9]:
# Summary beeswarm plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, show=False, max_display=15)
plt.tight_layout()
plt.savefig(OUT_DIR / "kernel_shap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "kernel_shap_summary_beeswarm.png")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/kernel_shap_summary_beeswarm.png


In [10]:
# Bar chart of mean |SHAP| importance
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, plot_type="bar", show=False, max_display=15)
plt.tight_layout()
plt.savefig(OUT_DIR / "kernel_shap_summary_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "kernel_shap_summary_bar.png")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/kernel_shap_summary_bar.png


### 5.2 Waterfall plot (single prediction explanation)

In [11]:
# Create SHAP Explanation object for waterfall plot
explanation = shap.Explanation(
    values=shap_values,
    base_values=np.full(len(shap_values), explainer.expected_value),
    data=X_sample,
    feature_names=FEATURE_COLS,
)

# Waterfall plot for first sample
idx = 0
pred = hybrid_predict_fn(X_sample[idx:idx+1])[0]
actual = y_sample[idx]

print(f"Sample {idx}: Actual=${actual:,.0f}, Predicted=${pred:,.0f}")
print(f"Base value (expected): ${explainer.expected_value:,.0f}")
print(f"Sum of SHAP values: ${shap_values[idx].sum():,.0f}")
print(f"Base + SHAP sum = ${explainer.expected_value + shap_values[idx].sum():,.0f}")

fig, ax = plt.subplots(figsize=(10, 8))
shap.waterfall_plot(explanation[idx], show=False, max_display=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "kernel_shap_waterfall_sample0.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "kernel_shap_waterfall_sample0.png")

Sample 0: Actual=$722,000, Predicted=$676,425
Base value (expected): $461,092
Sum of SHAP values: $215,333
Base + SHAP sum = $676,425
Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/kernel_shap_waterfall_sample0.png


## 6 — Save artifacts

In [12]:
# Save KernelExplainer and computed SHAP values
kernel_shap_payload = {
    "explainer_background": background.data,
    "expected_value": float(explainer.expected_value),
    "shap_values_sample": shap_values,
    "X_sample": X_sample,
    "y_sample": y_sample,
    "sample_indices": test_sample_idx,
    "n_shap_samples": N_SHAP_SAMPLES,
    "feature_columns": FEATURE_COLS,
}
joblib.dump(kernel_shap_payload, OUT_DIR / "kernel_shap_values.joblib")
print("Saved", OUT_DIR / "kernel_shap_values.joblib")

# Save global importance as JSON
kernel_shap_importance = {
    "method": "KernelSHAP",
    "model": "full_hybrid_ensemble",
    "n_samples": N_SAMPLES,
    "n_shap_samples": N_SHAP_SAMPLES,
    "expected_value": float(explainer.expected_value),
    "feature_importance": {k: float(v) for k, v in feature_importance.items()},
}
with open(OUT_DIR / "kernel_shap_global_importance.json", "w") as f:
    json.dump(kernel_shap_importance, f, indent=2)
print("Saved", OUT_DIR / "kernel_shap_global_importance.json")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/kernel_shap_values.joblib
Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/kernel_shap_global_importance.json


## 7 — Summary

KernelSHAP provides **exact** SHAP values for the full hybrid ensemble by treating it as a black-box. The trade-off is speed: each row takes ~1-5 minutes to explain.

**Artifacts created:**
- `kernel_shap_values.joblib` — Background data, SHAP values, sample data
- `kernel_shap_global_importance.json` — Global feature importance
- `kernel_shap_summary_*.png` — Visualization plots

**Use cases:**
- Validating faster approximate methods (Composite TreeSHAP)
- Single-row explanations where accuracy is critical
- Debugging unexpected predictions

In [13]:
print("\n── KernelSHAP Artifacts ──")
for p in sorted(OUT_DIR.glob("kernel_shap*")):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size/1024:,.1f} KB")


── KernelSHAP Artifacts ──
  kernel_shap_global_importance.json: 2.8 KB
  kernel_shap_summary_bar.png: 76.2 KB
  kernel_shap_summary_beeswarm.png: 106.2 KB
  kernel_shap_values.joblib: 51.6 KB
  kernel_shap_waterfall_sample0.png: 108.2 KB
